# GT-matching alignment test
ParkingLot2_008_overfence1 — cam_01 vs cam_03

In [27]:
import numpy as np
from pathlib import Path

SCENE = 'ParkingLot1_002_overfence2'
BASE  = Path(f'/iopsstor/scratch/cscs/tnanni/ghost_outputs/rich11_segmentation_test/{SCENE}')

def load_tracks(cam: str) -> dict:
    body_dir = BASE / cam / 'body_data'
    persons = {}
    for npz_path in sorted(body_dir.glob('person_*.npz')):
        pid = int(npz_path.stem.split('_')[1])
        with np.load(str(npz_path)) as d:
            if 'smplx_transl' not in d or 'frame_indices' not in d:
                print(f'{cam}/P{pid}: missing smplx_transl or frame_indices, skipping')
                continue
            persons[pid] = {
                'transl': d['smplx_transl'].copy(),
                'frames': d['frame_indices'].copy(),
            }
            if 'pred_keypoints_3d' in d:
                persons[pid]['kpts3d'] = d['pred_keypoints_3d'].copy()  # (T, J, 3)
        print(f'{cam}/P{pid}: {len(persons[pid]["frames"])} frames'
              + (f'  kpts3d shape={persons[pid]["kpts3d"].shape}' if 'kpts3d' in persons[pid] else '  NO kpts3d'))
    return persons

cam1 = load_tracks('cam_01')
cam3 = load_tracks('cam_03')
print(f'\ncam_01: {sorted(cam1.keys())}  cam_03: {sorted(cam3.keys())}')

cam_01/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P2: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P3: 234 frames  kpts3d shape=(234, 70, 3)
cam_01/P4: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P5: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P6: 270 frames  kpts3d shape=(270, 70, 3)
cam_03/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_03/P2: 270 frames  kpts3d shape=(270, 70, 3)
cam_03/P3: 248 frames  kpts3d shape=(248, 70, 3)
cam_03/P4: 173 frames  kpts3d shape=(173, 70, 3)
cam_03/P5: 198 frames  kpts3d shape=(198, 70, 3)

cam_01: [1, 2, 3, 4, 5, 6]  cam_03: [1, 2, 3, 4, 5]


In [28]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# MHR70 skeleton edges as (joint_a_idx, joint_b_idx)
MHR70_EDGES = [
    # Body
    (13, 11), (11, 9),          # left leg
    (14, 12), (12, 10),         # right leg
    (9, 10),                    # hips
    (5, 9),  (6, 10),           # torso sides
    (5, 6),                     # shoulders
    (69, 5), (69, 6),           # neck to shoulders
    (5, 7),  (7, 62),           # left arm
    (6, 8),  (8, 41),           # right arm
    (0, 1),  (0, 2), (1, 2),    # face triangle
    (1, 3),  (2, 4),            # eyes to ears
    (3, 5),  (4, 6),            # ears to shoulders
    # Feet
    (13, 15), (13, 16), (13, 17),
    (14, 18), (14, 19), (14, 20),
    # Left hand
    (62, 45), (45, 44), (44, 43), (43, 42),
    (62, 49), (49, 48), (48, 47), (47, 46),
    (62, 53), (53, 52), (52, 51), (51, 50),
    (62, 57), (57, 56), (56, 55), (55, 54),
    (62, 61), (61, 60), (60, 59), (59, 58),
    # Right hand
    (41, 24), (24, 23), (23, 22), (22, 21),
    (41, 28), (28, 27), (27, 26), (26, 25),
    (41, 32), (32, 31), (31, 30), (30, 29),
    (41, 36), (36, 35), (35, 34), (34, 33),
    (41, 40), (40, 39), (39, 38), (38, 37),
]

P1, P3 = 1, 1
FRAME_IDX = 0   # <-- change this to inspect different frames

def get_overlapping_kpts(data_a, data_b):
    fa = {int(fi): i for i, fi in enumerate(data_a['frames'])}
    fb = {int(fi): i for i, fi in enumerate(data_b['frames'])}
    common = sorted(set(fa) & set(fb))
    return (data_a['kpts3d'][[fa[fi] for fi in common]],
            data_b['kpts3d'][[fb[fi] for fi in common]])

kpts_c1, kpts_c3 = get_overlapping_kpts(cam1[P1], cam3[P3])

# Root-relative (joint 9 = left hip, use midpoint of hips as root)
root_c1 = ((kpts_c1[:, 9, :] + kpts_c1[:, 10, :]) / 2)[:, None, :]
root_c3 = ((kpts_c3[:, 9, :] + kpts_c3[:, 10, :]) / 2)[:, None, :]
kpts_c1_rel = kpts_c1 - root_c1
kpts_c3_rel = kpts_c3 - root_c3

T = len(kpts_c1_rel)
print(f'Overlapping frames: {T}   Showing frame index: {FRAME_IDX}')

frame_c1 = kpts_c1_rel[FRAME_IDX]   # (70, 3)
frame_c3 = kpts_c3_rel[FRAME_IDX]   # (70, 3)

def skeleton_traces(kpts, color, name, showlegend=True):
    traces = []
    # Joints
    traces.append(go.Scatter3d(
        x=kpts[:, 0], y=kpts[:, 1], z=kpts[:, 2],
        mode='markers',
        marker=dict(color=color, size=4),
        name=name,
        showlegend=showlegend,
    ))
    # Bones — pack all edges into one trace with None separators
    xs, ys, zs = [], [], []
    for a, b in MHR70_EDGES:
        xs += [kpts[a, 0], kpts[b, 0], None]
        ys += [kpts[a, 1], kpts[b, 1], None]
        zs += [kpts[a, 2], kpts[b, 2], None]
    traces.append(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='lines',
        line=dict(color=color, width=3),
        showlegend=False,
    ))
    return traces

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=[f'cam_01 / P{P1}', f'cam_03 / P{P3}'],
)

for tr in skeleton_traces(frame_c1, '#1f77b4', f'cam1/P{P1}'):
    fig.add_trace(tr, row=1, col=1)
for tr in skeleton_traces(frame_c3, '#d62728', f'cam3/P{P3}'):
    fig.add_trace(tr, row=1, col=2)

axis = dict(showgrid=True, zeroline=True)
fig.update_layout(
    title=f'MHR70 skeleton (hip-centred) — frame {FRAME_IDX}/{T-1}<br>cam1/P{P1} (blue)  cam3/P{P3} (red)',
    scene=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode='data'),
    scene2=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode='data'),
    height=650,
    margin=dict(l=0, r=0, b=0, t=60),
)
# fig.show()

Overlapping frames: 270   Showing frame index: 0


In [29]:
# GT correspondences — update these per scene
GT_PAIRS = [(1, 1), (2, 2), (4, 4), (6, 5)]   # <-- edit here

def affine_fit(src: np.ndarray, dst: np.ndarray):
    """Full affine transform (3x3 A + t) mapping src -> dst via least squares.
    src, dst: (N, 3)."""
    ones = np.ones((len(src), 1), dtype=src.dtype)
    X = np.concatenate([src, ones], axis=1)          # (N, 4)
    M, _, _, _ = np.linalg.lstsq(X, dst, rcond=None) # (4, 3)
    A = M[:3].T
    t = M[3]
    return A, t

def apply_affine(A, t, pts):
    return (A @ pts.T).T + t

def overlapping_kpts(data_a, data_b):
    """Return (kpts_a, kpts_b) at common frame indices. Shape: (T, 70, 3) each."""
    fa = {int(fi): i for i, fi in enumerate(data_a['frames'])}
    fb = {int(fi): i for i, fi in enumerate(data_b['frames'])}
    common = sorted(set(fa) & set(fb))
    if not common:
        return None, None
    ka = data_a['kpts3d'][[fa[fi] for fi in common]]  # (T, 70, 3)
    kb = data_b['kpts3d'][[fb[fi] for fi in common]]
    return ka, kb


all_src, all_dst = [], []
pair_slices = []   # track which flattened rows belong to which pair

for p1, p3 in GT_PAIRS:
    if p1 not in cam1 or p3 not in cam3:
        print(f'WARNING: c1p{p1} or c3p{p3} not found, skipping')
        continue
    if 'kpts3d' not in cam1[p1] or 'kpts3d' not in cam3[p3]:
        print(f'WARNING: c1p{p1} or c3p{p3} missing kpts3d, skipping')
        continue
    ka, kb = overlapping_kpts(cam1[p1], cam3[p3])
    if ka is None:
        print(f'WARNING: c1p{p1} ↔ c3p{p3}: no overlapping frames')
        continue
    T = len(ka)
    # Flatten: (T, 70, 3) → (T*70, 3) — joint j always maps to joint j
    src_flat = ka.reshape(-1, 3)
    dst_flat = kb.reshape(-1, 3)
    start = sum(len(s) for s in all_src)
    all_src.append(src_flat)
    all_dst.append(dst_flat)
    pair_slices.append((p1, p3, T, start, start + len(src_flat)))
    print(f'c1p{p1} ↔ c3p{p3}: {T} frames → {len(src_flat)} point pairs')

src_cat = np.concatenate(all_src)
dst_cat = np.concatenate(all_dst)
print(f'\nTotal point pairs: {len(src_cat)}')

A, t_vec = affine_fit(src_cat, dst_cat)
aligned_cat = apply_affine(A, t_vec, src_cat)
global_rmse = float(np.sqrt(((aligned_cat - dst_cat) ** 2).sum(1).mean()))

print(f'\nGlobal RMSE: {global_rmse:.4f} m')
print(f'A:\n{A}')
print(f't: {t_vec}')
print()

for p1, p3, T, start, end in pair_slices:
    aligned_chunk = apply_affine(A, t_vec, src_cat[start:end])
    rmse = float(np.sqrt(((aligned_chunk - dst_cat[start:end]) ** 2).sum(1).mean()))
    print(f'  c1p{p1} ↔ c3p{p3}:  {T} frames  RMSE = {rmse:.4f} m')

c1p1 ↔ c3p1: 270 frames → 18900 point pairs
c1p2 ↔ c3p2: 270 frames → 18900 point pairs
c1p4 ↔ c3p4: 173 frames → 12110 point pairs
c1p6 ↔ c3p5: 198 frames → 13860 point pairs

Total point pairs: 63770

Global RMSE: 0.1012 m
A:
[[ 0.88968974  0.06067393 -0.31992146]
 [-0.01031466  0.93487465  0.28826   ]
 [ 0.29498208 -0.17130624  0.80562335]]
t: [ 0.05270239 -0.05112822 -0.17154506]

  c1p1 ↔ c3p1:  270 frames  RMSE = 0.0659 m
  c1p2 ↔ c3p2:  270 frames  RMSE = 0.1147 m
  c1p4 ↔ c3p4:  173 frames  RMSE = 0.1105 m
  c1p6 ↔ c3p5:  198 frames  RMSE = 0.1124 m


In [30]:
import plotly.graph_objects as go

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
          '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf']

def add_track(fig, pts, color, name, dash='solid'):
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='lines',
        line=dict(color=color, width=3, dash=dash),
        name=name, legendgroup=name,
    ))
    fig.add_trace(go.Scatter3d(
        x=[pts[0, 0]], y=[pts[0, 1]], z=[pts[0, 2]],
        mode='markers', marker=dict(color=color, size=6, symbol='circle'),
        legendgroup=name, showlegend=False,
    ))

fig = go.Figure()

for i, (p1, p3) in enumerate(GT_PAIRS):
    if p1 not in cam1 or p3 not in cam3:
        continue
    color = COLORS[i % len(COLORS)]
    add_track(fig, cam3[p3]['transl'], color, f'cam3/P{p3}')
    aligned = apply_affine(A, t_vec, cam1[p1]['transl'])
    add_track(fig, aligned, color, f'cam1/P{p1} → cam3/P{p3}', dash='dash')

fig.update_layout(
    title=f'Affine alignment   global RMSE={global_rmse:.4f} m<br>solid=cam3, dashed=cam1 aligned',
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)', aspectmode='data'),
    legend=dict(itemsizing='constant'),
    margin=dict(l=0, r=0, b=0, t=60),
)
fig.show()

In [31]:
from scipy.optimize import linear_sum_assignment

MIN_OVERLAP = 30  # frames

def pose_cosine_sim(data_a, data_b):
    """Mean frame-by-frame cosine similarity on root-relative 3-D keypoints."""
    fa = {int(fi): i for i, fi in enumerate(data_a['frames'])}
    fb = {int(fi): i for i, fi in enumerate(data_b['frames'])}
    common = sorted(set(fa) & set(fb))
    if len(common) < MIN_OVERLAP:
        return np.nan
    ka = data_a['kpts3d'][[fa[fi] for fi in common]]   # (T, 70, 3)
    kb = data_b['kpts3d'][[fb[fi] for fi in common]]
    # hip-midpoint root-relative
    root_a = ((ka[:, 9, :] + ka[:, 10, :]) / 2)[:, None, :]
    root_b = ((kb[:, 9, :] + kb[:, 10, :]) / 2)[:, None, :]
    ka_rel = (ka - root_a).reshape(len(common), -1)    # (T, 210)
    kb_rel = (kb - root_b).reshape(len(common), -1)
    ka_n = ka_rel / (np.linalg.norm(ka_rel, axis=1, keepdims=True) + 1e-8)
    kb_n = kb_rel / (np.linalg.norm(kb_rel, axis=1, keepdims=True) + 1e-8)
    return float((ka_n * kb_n).sum(axis=1).mean())

pids1 = sorted(cam1.keys())
pids3 = sorted(cam3.keys())

sim = np.full((len(pids1), len(pids3)), np.nan)
for i, p1 in enumerate(pids1):
    for j, p3 in enumerate(pids3):
        if 'kpts3d' in cam1[p1] and 'kpts3d' in cam3[p3]:
            sim[i, j] = pose_cosine_sim(cam1[p1], cam3[p3])

# Print similarity matrix
header = "        " + "   ".join(f"c3/P{p}" for p in pids3)
print(header)
for i, p1 in enumerate(pids1):
    vals = "   ".join(f"{sim[i,j]:.3f}" if not np.isnan(sim[i,j]) else "  nan" for j in range(len(pids3)))
    print(f"c1/P{p1}:  {vals}")

# Hungarian matching (maximise similarity)
sim_for_match = np.where(np.isnan(sim), -1.0, sim)
row_ind, col_ind = linear_sum_assignment(-sim_for_match)

print("\nHungarian matching:")
for r, c in zip(row_ind, col_ind):
    tag = "✓ GT" if (pids1[r], pids3[c]) in GT_PAIRS else "✗ WRONG"
    print(f"  cam1/P{pids1[r]} → cam3/P{pids3[c]}   sim={sim[r,c]:.3f}  {tag}")

        c3/P1   c3/P2   c3/P3   c3/P4   c3/P5
c1/P1:  0.948   0.533   0.498   0.513   0.501
c1/P2:  0.502   0.916   0.682   0.596   0.686
c1/P3:  0.491   0.796   0.749   0.435   0.479
c1/P4:  0.514   0.530   0.381   0.913   0.730
c1/P5:  0.405   0.696   0.608   0.487   0.595
c1/P6:  0.535   0.644   0.514   0.620   0.921

Hungarian matching:
  cam1/P1 → cam3/P1   sim=0.948  ✓ GT
  cam1/P2 → cam3/P2   sim=0.916  ✓ GT
  cam1/P3 → cam3/P3   sim=0.749  ✗ WRONG
  cam1/P4 → cam3/P4   sim=0.913  ✓ GT
  cam1/P6 → cam3/P5   sim=0.921  ✓ GT


In [32]:
import plotly.graph_objects as go

labels_x = [f"c3/P{p}" for p in pids3]
labels_y = [f"c1/P{p}" for p in pids1]

fig = go.Figure(go.Heatmap(
    z=sim,
    x=labels_x, y=labels_y,
    colorscale='RdBu', zmid=0,
    text=[[f"{sim[i,j]:.3f}" if not np.isnan(sim[i,j]) else "nan"
           for j in range(len(pids3))] for i in range(len(pids1))],
    texttemplate='%{text}',
    showscale=True,
))
fig.update_layout(
    title='Pose cosine similarity — root-relative kpts3d (cam_01 rows × cam_03 cols)',
    height=400,
    xaxis_title='cam_03', yaxis_title='cam_01',
)
fig.show()

In [33]:
# Discover and load all cameras that have body_data
all_tracks = {}
for cam_dir in sorted(BASE.iterdir()):
    body_dir = cam_dir / 'body_data'
    if not body_dir.exists():
        continue
    cam = cam_dir.name
    tracks = load_tracks(cam)
    if tracks:
        all_tracks[cam] = tracks

print(f'\nLoaded {len(all_tracks)} cameras: {sorted(all_tracks.keys())}')
print({cam: sorted(p.keys()) for cam, p in all_tracks.items()})

cam_00/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P2: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P3: 234 frames  kpts3d shape=(234, 70, 3)
cam_01/P4: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P5: 270 frames  kpts3d shape=(270, 70, 3)
cam_01/P6: 270 frames  kpts3d shape=(270, 70, 3)
cam_02/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_03/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_03/P2: 270 frames  kpts3d shape=(270, 70, 3)
cam_03/P3: 248 frames  kpts3d shape=(248, 70, 3)
cam_03/P4: 173 frames  kpts3d shape=(173, 70, 3)
cam_03/P5: 198 frames  kpts3d shape=(198, 70, 3)
cam_04/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_04/P4: 270 frames  kpts3d shape=(270, 70, 3)
cam_05/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_06/P1: 270 frames  kpts3d shape=(270, 70, 3)
cam_06/P4: 270 frames  kpts3d shape=(270, 70, 3)
cam_07/P1: 269 frames  kpts3d shape=(269, 70, 3)

Loaded 8 cameras: ['cam_00', 'cam_01', 'cam_02', 'cam_03', 'cam_04',

In [34]:
from itertools import combinations, permutations
from collections import defaultdict

MIN_PAIR_OVERLAP  = 30   # skip a pair if it has fewer overlapping frames than this
MIN_PAIRS_FOR_FIT = 1    # need at least this many contributing pairs to trust the fit

def overlapping_kpts_for_fit(data_a, data_b):
    fa = {int(fi): i for i, fi in enumerate(data_a['frames'])}
    fb = {int(fi): i for i, fi in enumerate(data_b['frames'])}
    common = sorted(set(fa) & set(fb))
    if len(common) < MIN_PAIR_OVERLAP:
        return None, None
    ka = data_a['kpts3d'][[fa[fi] for fi in common]]   # (T, 70, 3)
    kb = data_b['kpts3d'][[fb[fi] for fi in common]]
    return ka, kb

def score_assignment(assignment, persons_a, persons_b):
    """Fit one affine transform to all paired keypoints; return RMSE."""
    all_src, all_dst = [], []
    for pid_a, pid_b in assignment:
        ka, kb = overlapping_kpts_for_fit(persons_a[pid_a], persons_b[pid_b])
        if ka is None:
            continue
        all_src.append(ka.reshape(-1, 3))
        all_dst.append(kb.reshape(-1, 3))
    if len(all_src) < MIN_PAIRS_FOR_FIT:
        return float('inf')
    src = np.concatenate(all_src)
    dst = np.concatenate(all_dst)
    A, t = affine_fit(src, dst)
    aligned = apply_affine(A, t, src)
    return float(np.sqrt(((aligned - dst) ** 2).sum(1).mean()))

def find_best_assignment(persons_a, persons_b):
    pids_a = sorted(persons_a.keys())
    pids_b = sorted(persons_b.keys())
    n_match = min(len(pids_a), len(pids_b))

    best_rmse, best_pairs = float('inf'), None
    for subset_a in combinations(range(len(pids_a)), n_match):
        for perm_b in permutations(range(len(pids_b))):
            assignment = [(pids_a[i], pids_b[j]) for i, j in zip(subset_a, perm_b)]
            rmse = score_assignment(assignment, persons_a, persons_b)
            if rmse < best_rmse:
                best_rmse, best_pairs = rmse, assignment
    return best_pairs, best_rmse

# --- Union-Find ---
class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[ry] = rx

uf = UnionFind()
for cam, persons in all_tracks.items():
    for pid in persons:
        uf.find((cam, pid))

cam_list = sorted(all_tracks.keys())

RMSE_THRESHOLD = 0.20   # m — if the best assignment still has RMSE above this, skip the pair

for cam_a, cam_b in combinations(cam_list, 2):
    persons_a, persons_b = all_tracks[cam_a], all_tracks[cam_b]
    best_pairs, best_rmse = find_best_assignment(persons_a, persons_b)

    print(f'\n{cam_a} × {cam_b}   best RMSE={best_rmse:.3f} m:')
    if best_pairs is None:
        print('  [not enough overlapping data]')
        continue
    for pid_a, pid_b in best_pairs:
        ka, kb = overlapping_kpts_for_fit(persons_a[pid_a], persons_b[pid_b])
        n_frames = len(ka) if ka is not None else 0
        if best_rmse > RMSE_THRESHOLD:
            print(f'  P{pid_a} → P{pid_b}  ({n_frames} frames)  [skipped: RMSE too high]')
            continue
        print(f'  P{pid_a} → P{pid_b}  ({n_frames} frames)  [merged]')
        uf.union((cam_a, pid_a), (cam_b, pid_b))

# Build global IDs
clusters = defaultdict(list)
for cam in cam_list:
    for pid in all_tracks[cam]:
        clusters[uf.find((cam, pid))].append((cam, pid))

global_id = {}
for gid, (_, members) in enumerate(sorted(clusters.items()), 1):
    for node in members:
        global_id[node] = gid

# Cluster summary
by_gid = defaultdict(list)
for node, gid in global_id.items():
    by_gid[gid].append(node)

print('\n=== Clusters ===')
for gid in sorted(by_gid.keys()):
    members = sorted(by_gid[gid])
    label = ',  '.join(f'{cam}/P{pid}' for cam, pid in members)
    print(f'  Person {gid}: {label}')


cam_00 × cam_01   best RMSE=0.059 m:
  P1 → P1  (270 frames)  [merged]

cam_00 × cam_02   best RMSE=0.119 m:
  P1 → P1  (270 frames)  [merged]

cam_00 × cam_03   best RMSE=0.062 m:
  P1 → P1  (270 frames)  [merged]

cam_00 × cam_04   best RMSE=0.074 m:
  P1 → P1  (270 frames)  [merged]

cam_00 × cam_05   best RMSE=0.071 m:
  P1 → P1  (270 frames)  [merged]

cam_00 × cam_06   best RMSE=0.074 m:
  P1 → P1  (270 frames)  [merged]

cam_00 × cam_07   best RMSE=0.073 m:
  P1 → P1  (269 frames)  [merged]

cam_01 × cam_02   best RMSE=0.111 m:
  P1 → P1  (270 frames)  [merged]

cam_01 × cam_03   best RMSE=0.172 m:
  P1 → P1  (270 frames)  [merged]
  P2 → P2  (270 frames)  [merged]
  P3 → P3  (230 frames)  [merged]
  P4 → P4  (173 frames)  [merged]
  P6 → P5  (198 frames)  [merged]

cam_01 × cam_04   best RMSE=0.127 m:
  P1 → P1  (270 frames)  [merged]
  P4 → P4  (270 frames)  [merged]

cam_01 × cam_05   best RMSE=0.070 m:
  P1 → P1  (270 frames)  [merged]

cam_01 × cam_06   best RMSE=0.110 m:


In [35]:
from itertools import combinations
from collections import defaultdict

# ── ICP ────────────────────────────────────────────────────────────────────────
def icp(src_kpts, dst_kpts, n_iters=30, tol=1e-5):
    """
    Find affine T: cam_b space → cam_a space with NO temporal correspondence.
    For each joint k, each src frame finds its nearest dst frame in joint-k space.
    src_kpts: (T_s, J, 3)   dst_kpts: (T_d, J, 3)
    Returns: A (3,3), t_vec (3,), final RMSE
    """
    A, t_vec = np.eye(3), np.zeros(3)
    J = src_kpts.shape[1]
    prev_rmse = np.inf

    for _ in range(n_iters):
        src_t = (A @ src_kpts.reshape(-1, 3).T).T.reshape(src_kpts.shape)

        nn_src, nn_dst, sq_errs = [], [], []
        for k in range(J):
            s = src_t[:, k, :]       # (T_s, 3)
            d = dst_kpts[:, k, :]    # (T_d, 3)
            d2 = ((s[:, None] - d[None]) ** 2).sum(-1)  # (T_s, T_d)
            idx = d2.argmin(1)
            nn_src.append(src_kpts[:, k, :])
            nn_dst.append(d[idx])
            sq_errs.append(d2[np.arange(len(s)), idx])

        rmse = float(np.sqrt(np.concatenate(sq_errs).mean()))
        if abs(prev_rmse - rmse) < tol:
            break
        prev_rmse = rmse
        A, t_vec = affine_fit(np.concatenate(nn_src), np.concatenate(nn_dst))

    return A, t_vec, rmse


def apply_T(A, t_vec, kpts):
    """Apply affine transform to (T, J, 3) keypoints."""
    shape = kpts.shape
    return (A @ kpts.reshape(-1, 3).T).T.reshape(shape) + t_vec


def nn_rmse(kpts_transformed, kpts_ref):
    """Per-joint NN RMSE between already-transformed src and ref (T is fixed)."""
    sq_errs = []
    for k in range(kpts_transformed.shape[1]):
        s = kpts_transformed[:, k, :]
        d = kpts_ref[:, k, :]
        d2 = ((s[:, None] - d[None]) ** 2).sum(-1)
        sq_errs.append(d2.min(1))
    return float(np.sqrt(np.concatenate(sq_errs).mean()))


# ── Union-Find ─────────────────────────────────────────────────────────────────
class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[ry] = rx

uf = UnionFind()
for cam, persons in all_tracks.items():
    for pid in persons:
        uf.find((cam, pid))

cam_list = sorted(all_tracks.keys())

ANCHOR_RMSE_THR = 0.15  # m — skip pair if best anchor ICP RMSE > this
MATCH_RMSE_THR  = 0.25  # m — threshold for remaining track matches after T is fixed

for cam_a, cam_b in combinations(cam_list, 2):
    persons_a = all_tracks[cam_a]
    persons_b = all_tracks[cam_b]

    # Always register the smaller set (cam_b) into the larger one (cam_a)
    if len(persons_b) > len(persons_a):
        cam_a, cam_b = cam_b, cam_a
        persons_a, persons_b = persons_b, persons_a

    valid_b = {pid: d for pid, d in persons_b.items() if 'kpts3d' in d}
    valid_a = {pid: d for pid, d in persons_a.items() if 'kpts3d' in d}
    if not valid_b or not valid_a:
        continue

    # ── Step 1: anchor = longest track in cam_b ────────────────────────────────
    anchor_pid = max(valid_b, key=lambda pid: len(valid_b[pid]['frames']))
    anchor_kpts = valid_b[anchor_pid]['kpts3d']   # (T_b, J, 3)

    # ICP anchor against every cam_a track → pick best match → gives global T
    best = {'rmse': np.inf, 'A': None, 't': None, 'pid_a': None}
    for pid_a, data_a in valid_a.items():
        A, t_vec, rmse = icp(anchor_kpts, data_a['kpts3d'])
        if rmse < best['rmse']:
            best = {'rmse': rmse, 'A': A, 't': t_vec, 'pid_a': pid_a}

    print(f'\n{cam_a} × {cam_b}   anchor: {cam_b}/P{anchor_pid} → {cam_a}/P{best["pid_a"]}  ICP RMSE={best["rmse"]:.3f} m')

    if best['rmse'] > ANCHOR_RMSE_THR:
        print('  [skipped: anchor RMSE too high]')
        continue

    A, t_vec = best['A'], best['t']
    matched_a = {best['pid_a']}
    matched_b = {anchor_pid}
    uf.union((cam_a, best['pid_a']), (cam_b, anchor_pid))
    print(f'  P{anchor_pid} → P{best["pid_a"]}  [anchor merged]')

    # ── Step 2: remaining cam_b tracks — apply global T, match by NN RMSE ──────
    remaining_a = {pid: d for pid, d in valid_a.items() if pid not in matched_a}
    for pid_b, data_b in valid_b.items():
        if pid_b in matched_b:
            continue
        kpts_b_t = apply_T(A, t_vec, data_b['kpts3d'])

        best_m = {'rmse': np.inf, 'pid_a': None}
        for pid_a, data_a in remaining_a.items():
            rmse = nn_rmse(kpts_b_t, data_a['kpts3d'])
            if rmse < best_m['rmse']:
                best_m = {'rmse': rmse, 'pid_a': pid_a}

        if best_m['pid_a'] is not None and best_m['rmse'] < MATCH_RMSE_THR:
            print(f'  P{pid_b} → P{best_m["pid_a"]}  RMSE={best_m["rmse"]:.3f}  [merged]')
            uf.union((cam_a, best_m['pid_a']), (cam_b, pid_b))
            matched_a.add(best_m['pid_a'])
            matched_b.add(pid_b)
            del remaining_a[best_m['pid_a']]
        else:
            print(f'  P{pid_b} → P{best_m["pid_a"]}  RMSE={best_m["rmse"]:.3f}  [skipped]')

# ── Global ID assignment ────────────────────────────────────────────────────────
clusters = defaultdict(list)
for cam in sorted(all_tracks.keys()):
    for pid in all_tracks[cam]:
        clusters[uf.find((cam, pid))].append((cam, pid))

global_id = {}
for gid, (_, members) in enumerate(sorted(clusters.items()), 1):
    for node in members:
        global_id[node] = gid

by_gid = defaultdict(list)
for node, gid in global_id.items():
    by_gid[gid].append(node)

print('\n=== Clusters ===')
for gid in sorted(by_gid.keys()):
    members = sorted(by_gid[gid])
    print(f'  Person {gid}: {",  ".join(f"{c}/P{p}" for c, p in members)}')


cam_01 × cam_00   anchor: cam_00/P1 → cam_01/P1  ICP RMSE=0.142 m
  P1 → P1  [anchor merged]

cam_00 × cam_02   anchor: cam_02/P1 → cam_00/P1  ICP RMSE=0.115 m
  P1 → P1  [anchor merged]

cam_03 × cam_00   anchor: cam_00/P1 → cam_03/P1  ICP RMSE=0.129 m
  P1 → P1  [anchor merged]

cam_04 × cam_00   anchor: cam_00/P1 → cam_04/P1  ICP RMSE=0.135 m
  P1 → P1  [anchor merged]

cam_00 × cam_05   anchor: cam_05/P1 → cam_00/P1  ICP RMSE=0.104 m
  P1 → P1  [anchor merged]

cam_06 × cam_00   anchor: cam_00/P1 → cam_06/P1  ICP RMSE=0.148 m
  P1 → P1  [anchor merged]

cam_00 × cam_07   anchor: cam_07/P1 → cam_00/P1  ICP RMSE=0.117 m
  P1 → P1  [anchor merged]

cam_01 × cam_02   anchor: cam_02/P1 → cam_01/P1  ICP RMSE=0.148 m
  P1 → P1  [anchor merged]

cam_01 × cam_03   anchor: cam_03/P1 → cam_01/P1  ICP RMSE=0.083 m
  P1 → P1  [anchor merged]
  P2 → P2  RMSE=0.056  [merged]
  P3 → P3  RMSE=0.205  [merged]
  P4 → P4  RMSE=0.146  [merged]
  P5 → P6  RMSE=0.087  [merged]

cam_01 × cam_04   anchor:

In [36]:

from scipy.optimize import linear_sum_assignment

# ── Thresholds ──────────────────────────────────────────────────────────────────
MIN_RANSAC_OVERLAP = 30   # min temporally-overlapping frames to attempt a fit
INLIER_THR         = 0.25  # m — track is an inlier if direct RMSE < this after RANSAC
RANSAC_ANCHOR_THR  = 0.20  # m — reject camera pair if best anchor RMSE > this
MATCH_RMSE_THR     = 0.25  # m — final merge threshold (Hungarian result)
DELTA_SEARCH       = 2     # frames — neighbourhood searched during δ refinement


# ── Helpers ─────────────────────────────────────────────────────────────────────

def get_aligned_slices(src_kpts, dst_kpts, delta):
    """Temporally aligned slices: frame i of src ↔ frame i+delta of dst."""
    T_s, T_d = len(src_kpts), len(dst_kpts)
    i0 = max(0, -delta)
    i1 = min(T_s, T_d - delta)
    if i1 - i0 < MIN_RANSAC_OVERLAP:
        return None, None
    return src_kpts[i0:i1], dst_kpts[i0 + delta : i1 + delta]


def fit_pair_delta(src_kpts, dst_kpts, delta):
    """Fit affine T on temporally-aligned keypoints. Returns (A, t, rmse) or None."""
    src_sl, dst_sl = get_aligned_slices(src_kpts, dst_kpts, delta)
    if src_sl is None:
        return None
    src_flat, dst_flat = src_sl.reshape(-1, 3), dst_sl.reshape(-1, 3)
    A, t_vec = affine_fit(src_flat, dst_flat)
    rmse = float(np.sqrt(((apply_affine(A, t_vec, src_flat) - dst_flat) ** 2).sum(1).mean()))
    return A, t_vec, rmse


def direct_rmse_at_delta(A, t_vec, src_kpts, dst_kpts, delta):
    """RMSE with T already fixed; frame i maps to frame i+delta (no NN in time)."""
    src_sl, dst_sl = get_aligned_slices(src_kpts, dst_kpts, delta)
    if src_sl is None:
        return float('inf')
    return float(np.sqrt(((apply_T(A, t_vec, src_sl) - dst_sl) ** 2).sum(-1).mean()))


def get_inlier_pairs(A, t_vec, delta, valid_b, valid_a, anchor_pid_b):
    """Return (pid_b, pid_a, rmse) for every non-anchor cam_b track that is
    a geometric inlier under the current T+δ (best match < INLIER_THR)."""
    inliers = []
    for pid_b, data_b in valid_b.items():
        if pid_b == anchor_pid_b:
            continue
        best = min(
            ((direct_rmse_at_delta(A, t_vec, data_b['kpts3d'], d['kpts3d'], delta), pid_a)
             for pid_a, d in valid_a.items()),
            key=lambda x: x[0]
        )
        if best[0] < INLIER_THR:
            inliers.append((pid_b, best[1], best[0]))
    return inliers


# ── Step 2: joint refinement ────────────────────────────────────────────────────

def joint_refine(assignment, valid_b, valid_a, delta_init):
    """Co-evolve T and δ using ALL assigned pairs simultaneously.
    assignment: list of (pid_b, pid_a).
    Returns: A, t_vec, delta, final_rmse
    """
    delta    = delta_init
    A        = np.eye(3)
    t_vec    = np.zeros(3)
    prev_rmse = float('inf')

    for _ in range(20):
        # --- fit T from all assigned pairs at current δ ---
        all_src, all_dst = [], []
        for pid_b, pid_a in assignment:
            src_sl, dst_sl = get_aligned_slices(
                valid_b[pid_b]['kpts3d'], valid_a[pid_a]['kpts3d'], delta)
            if src_sl is None:
                continue
            all_src.append(src_sl.reshape(-1, 3))
            all_dst.append(dst_sl.reshape(-1, 3))
        if not all_src:
            break
        A, t_vec = affine_fit(np.concatenate(all_src), np.concatenate(all_dst))

        # --- refine δ: search neighbourhood with current T ---
        best_d_rmse, best_d = float('inf'), delta
        for d in range(delta - DELTA_SEARCH, delta + DELTA_SEARCH + 1):
            sl_src, sl_dst = [], []
            for pid_b, pid_a in assignment:
                src_sl, dst_sl = get_aligned_slices(
                    valid_b[pid_b]['kpts3d'], valid_a[pid_a]['kpts3d'], d)
                if src_sl is None:
                    continue
                sl_src.append(apply_T(A, t_vec, src_sl).reshape(-1, 3))
                sl_dst.append(dst_sl.reshape(-1, 3))
            if not sl_src:
                continue
            rmse = float(np.sqrt(
                ((np.concatenate(sl_src) - np.concatenate(sl_dst)) ** 2).sum(1).mean()))
            if rmse < best_d_rmse:
                best_d_rmse, best_d = rmse, d

        if abs(prev_rmse - best_d_rmse) < 1e-5:
            break
        prev_rmse = best_d_rmse
        delta = best_d

    return A, t_vec, delta, best_d_rmse


# ── Main: RANSAC → joint refinement → Hungarian → Union-Find ───────────────────

uf_final = UnionFind()
for cam in sorted(all_tracks.keys()):
    for pid in all_tracks[cam]:
        uf_final.find((cam, pid))

for cam_a, cam_b in combinations(cam_list, 2):
    persons_a, persons_b = all_tracks[cam_a], all_tracks[cam_b]
    if len(persons_b) > len(persons_a):
        cam_a, cam_b = cam_b, cam_a
        persons_a, persons_b = persons_b, persons_a

    valid_b = {pid: d for pid, d in persons_b.items() if 'kpts3d' in d}
    valid_a = {pid: d for pid, d in persons_a.items() if 'kpts3d' in d}
    if not valid_b or not valid_a:
        continue

    # ── Step 1: RANSAC — find best anchor + δ ──────────────────────────────────
    best = {'inliers': -1, 'rmse': float('inf'),
            'A': None, 't': None, 'delta': 0, 'pid_b': None, 'pid_a': None}

    for pid_b, data_b in valid_b.items():
        src_kpts = data_b['kpts3d']
        T_b = len(src_kpts)
        for pid_a, data_a in valid_a.items():
            dst_kpts = data_a['kpts3d']
            T_a = len(dst_kpts)
            # δ scan
            bd = {'rmse': float('inf'), 'A': None, 't': None, 'delta': 0}
            for delta in range(-(T_b - 1), T_a):
                res = fit_pair_delta(src_kpts, dst_kpts, delta)
                if res and res[2] < bd['rmse']:
                    bd = {'rmse': res[2], 'A': res[0], 't': res[1], 'delta': delta}
            if bd['A'] is None:
                continue
            n_in = len(get_inlier_pairs(bd['A'], bd['t'], bd['delta'], valid_b, valid_a, pid_b))
            if (n_in > best['inliers'] or
                    (n_in == best['inliers'] and bd['rmse'] < best['rmse'])):
                best = {**bd, 'inliers': n_in, 'pid_b': pid_b, 'pid_a': pid_a}

    if best['A'] is None or best['rmse'] > RANSAC_ANCHOR_THR:
        print(f'\n{cam_a} × {cam_b}:  [RANSAC failed — best RMSE={best["rmse"]:.3f} m]')
        continue

    print(f'\n{cam_a} × {cam_b}:  RANSAC anchor {cam_b}/P{best["pid_b"]} → '
          f'{cam_a}/P{best["pid_a"]}  δ={best["delta"]}  '
          f'RMSE={best["rmse"]:.3f} m  inliers={best["inliers"]}')

    # ── Step 2: joint refinement — co-evolve T and δ with all inlier pairs ─────
    inlier_pairs = get_inlier_pairs(
        best['A'], best['t'], best['delta'], valid_b, valid_a, best['pid_b'])
    init_assignment = [(best['pid_b'], best['pid_a'])] + \
                      [(pb, pa) for pb, pa, _ in inlier_pairs]

    A, t_vec, delta, ref_rmse = joint_refine(init_assignment, valid_b, valid_a, best['delta'])
    print(f'  → after joint refinement: δ={delta}  RMSE={ref_rmse:.3f} m'
          f'  (used {len(init_assignment)} pairs)')

    # ── Step 3: Hungarian — optimal bijection on full RMSE matrix ──────────────
    pids_b = sorted(valid_b.keys())
    pids_a = sorted(valid_a.keys())
    rmse_mat = np.array([[
        direct_rmse_at_delta(A, t_vec, valid_b[pb]['kpts3d'], valid_a[pa]['kpts3d'], delta)
        for pa in pids_a] for pb in pids_b])

    rmse_for_lap = np.where(np.isinf(rmse_mat), 1e6, rmse_mat)
    row_ind, col_ind = linear_sum_assignment(rmse_for_lap)

    # ── Step 4: merge below threshold → Union-Find ─────────────────────────────
    for r, c in zip(row_ind, col_ind):
        pid_b, pid_a, rmse_val = pids_b[r], pids_a[c], rmse_mat[r, c]
        if rmse_val < MATCH_RMSE_THR:
            print(f'  P{pid_b} → P{pid_a}  RMSE={rmse_val:.3f}  [merged]')
            uf_final.union((cam_a, pid_a), (cam_b, pid_b))
        else:
            print(f'  P{pid_b} → P{pid_a}  RMSE={rmse_val:.3f}  [absent, skipped]')

# ── Cluster summary ─────────────────────────────────────────────────────────────
by_gid = defaultdict(list)
for cam in sorted(all_tracks.keys()):
    for pid in all_tracks[cam]:
        by_gid[uf_final.find((cam, pid))].append((cam, pid))

print('\n=== Final Clusters ===')
for gid, (_, members) in enumerate(sorted(by_gid.items()), 1):
    print(f'  Person {gid}: {",  ".join(f"{c}/P{p}" for c, p in sorted(members))}')



cam_01 × cam_00:  RANSAC anchor cam_00/P1 → cam_01/P1  δ=0  RMSE=0.059 m  inliers=0
  → after joint refinement: δ=0  RMSE=0.059 m  (used 1 pairs)
  P1 → P1  RMSE=0.059  [merged]

cam_00 × cam_02:  RANSAC anchor cam_02/P1 → cam_00/P1  δ=0  RMSE=0.124 m  inliers=0
  → after joint refinement: δ=0  RMSE=0.124 m  (used 1 pairs)
  P1 → P1  RMSE=0.124  [merged]

cam_03 × cam_00:  RANSAC anchor cam_00/P1 → cam_03/P1  δ=0  RMSE=0.062 m  inliers=0
  → after joint refinement: δ=0  RMSE=0.062 m  (used 1 pairs)
  P1 → P1  RMSE=0.062  [merged]

cam_04 × cam_00:  RANSAC anchor cam_00/P1 → cam_04/P1  δ=0  RMSE=0.074 m  inliers=0
  → after joint refinement: δ=0  RMSE=0.074 m  (used 1 pairs)
  P1 → P1  RMSE=0.074  [merged]

cam_00 × cam_05:  RANSAC anchor cam_05/P1 → cam_00/P1  δ=0  RMSE=0.069 m  inliers=0
  → after joint refinement: δ=0  RMSE=0.069 m  (used 1 pairs)
  P1 → P1  RMSE=0.069  [merged]

cam_06 × cam_00:  RANSAC anchor cam_00/P1 → cam_06/P1  δ=0  RMSE=0.074 m  inliers=0
  → after joint refi

: 